# RetainIQ — Phase 3.3: MySQL Validation & Analytical SQL

## Objective

I validate the populated MySQL model and use it to answer the first business questions. This notebook is query-heavy because I want Phase 3 to demonstrate real analytical SQL rather than stopping at table creation.

## 1. Load MySQL Query Helper

In [1]:
from getpass import getpass
import pandas as pd
import mysql.connector

MYSQL_CONFIG={"host":"localhost","port":3306,"user":"retainiq_user","password":getpass("Enter MySQL password for retainiq_user: "),"database":"retainiq"}

def get_connection(): return mysql.connector.connect(**MYSQL_CONFIG)
def run_query(query,params=None):
    connection=cursor=None
    try:
        connection=get_connection(); cursor=connection.cursor(dictionary=True); cursor.execute(query,params or ()); return pd.DataFrame(cursor.fetchall())
    finally:
        if cursor: cursor.close()
        if connection and connection.is_connected(): connection.close()
print("MySQL query helper ready.")

MySQL query helper ready.


## 2. Validate Row Counts

In [2]:
row_count_query="""SELECT 'fact_customer_status' table_name,COUNT(*) row_count FROM fact_customer_status UNION ALL SELECT 'dim_demographics',COUNT(*) FROM dim_demographics UNION ALL SELECT 'dim_location',COUNT(*) FROM dim_location UNION ALL SELECT 'dim_services',COUNT(*) FROM dim_services UNION ALL SELECT 'dim_account',COUNT(*) FROM dim_account UNION ALL SELECT 'dim_churn_detail',COUNT(*) FROM dim_churn_detail;"""
row_counts=run_query(row_count_query)
row_counts

,table_name,row_count
0,fact_customer_status,7043
1,dim_demographics,7043
2,dim_location,7043
3,dim_services,7043
4,dim_account,7043
5,dim_churn_detail,7043


In [3]:
assert (row_counts["row_count"]==7043).all()
print("PASS — Every analytical table contains 7,043 customer rows.")

PASS — Every analytical table contains 7,043 customer rows.


## 3. Duplicate Fact-Key Check

In [4]:
duplicate_keys=run_query("""SELECT customer_id,COUNT(*) row_count FROM fact_customer_status GROUP BY customer_id HAVING COUNT(*)>1;""")
duplicate_keys

""


In [5]:
assert duplicate_keys.empty
print("PASS — No duplicate customer IDs in fact_customer_status.")

PASS — No duplicate customer IDs in fact_customer_status.


## 4. Orphan Check

In [6]:
orphans=run_query("""SELECT 'demographics' dimension,COUNT(*) orphan_rows FROM dim_demographics d LEFT JOIN fact_customer_status f ON d.customer_id=f.customer_id WHERE f.customer_id IS NULL UNION ALL SELECT 'location',COUNT(*) FROM dim_location d LEFT JOIN fact_customer_status f ON d.customer_id=f.customer_id WHERE f.customer_id IS NULL UNION ALL SELECT 'services',COUNT(*) FROM dim_services d LEFT JOIN fact_customer_status f ON d.customer_id=f.customer_id WHERE f.customer_id IS NULL UNION ALL SELECT 'account',COUNT(*) FROM dim_account d LEFT JOIN fact_customer_status f ON d.customer_id=f.customer_id WHERE f.customer_id IS NULL UNION ALL SELECT 'churn_detail',COUNT(*) FROM dim_churn_detail d LEFT JOIN fact_customer_status f ON d.customer_id=f.customer_id WHERE f.customer_id IS NULL;""")
orphans

,dimension,orphan_rows
0,demographics,0
1,location,0
2,services,0
3,account,0
4,churn_detail,0


In [7]:
assert (orphans["orphan_rows"]==0).all()
print("PASS — No orphaned dimension rows.")

PASS — No orphaned dimension rows.


## 5. Headline KPI Reconciliation

In [8]:
headline=run_query("""SELECT COUNT(*) total_customers,SUM(churn_label='Yes') churned_customers,SUM(churn_label='No') non_churned_customers,ROUND(100*SUM(churn_label='Yes')/COUNT(*),2) churn_rate_pct FROM fact_customer_status;""")
headline

,total_customers,churned_customers,non_churned_customers,churn_rate_pct
0,7043,1869,5174,26.54


In [9]:
assert int(headline.loc[0,"total_customers"])==7043
assert int(headline.loc[0,"churned_customers"])==1869
assert round(float(headline.loc[0,"churn_rate_pct"]),1)==26.5
print("PASS — SQL reconciles to 7,043 customers, 1,869 churned, 26.5% churn.")

PASS — SQL reconciles to 7,043 customers, 1,869 churned, 26.5% churn.


## 6. Revenue at Risk

In [10]:
revenue_at_risk=run_query("""SELECT SUM(CASE WHEN churn_label='Yes' THEN total_revenue ELSE 0 END) revenue_at_risk FROM fact_customer_status;""")
revenue_at_risk

,revenue_at_risk
0,3684459.82


## 7. Churn by Contract

In [11]:
contract_analysis=run_query("""SELECT a.contract,COUNT(*) customers,SUM(f.churn_label='Yes') churned_customers,ROUND(100*SUM(f.churn_label='Yes')/COUNT(*),2) churn_rate_pct,SUM(CASE WHEN f.churn_label='Yes' THEN f.total_revenue ELSE 0 END) revenue_at_risk FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id GROUP BY a.contract ORDER BY churn_rate_pct DESC;""")
contract_analysis

,contract,customers,churned_customers,churn_rate_pct,revenue_at_risk
0,Month-to-Month,3610,1655,45.84,2490105.85
1,One Year,1550,166,10.71,858489.80
2,Two Year,1883,48,2.55,335864.17


## 8. Churn by Payment Method

In [12]:
payment_analysis=run_query("""SELECT a.payment_method,COUNT(*) customers,SUM(f.churn_label='Yes') churned_customers,ROUND(100*SUM(f.churn_label='Yes')/COUNT(*),2) churn_rate_pct FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id GROUP BY a.payment_method ORDER BY churn_rate_pct DESC;""")
payment_analysis

,payment_method,customers,churned_customers,churn_rate_pct
0,Mailed Check,385,142,36.88
1,Bank Withdrawal,3909,1329,34.00
2,Credit Card,2749,398,14.48


## 9. Churn by Internet Type

In [13]:
internet_analysis=run_query("""SELECT s.internet_type,COUNT(*) customers,SUM(f.churn_label='Yes') churned_customers,ROUND(100*SUM(f.churn_label='Yes')/COUNT(*),2) churn_rate_pct,SUM(CASE WHEN f.churn_label='Yes' THEN f.total_revenue ELSE 0 END) revenue_at_risk FROM fact_customer_status f JOIN dim_services s ON f.customer_id=s.customer_id GROUP BY s.internet_type ORDER BY churn_rate_pct DESC;""")
internet_analysis

,internet_type,customers,churned_customers,churn_rate_pct,revenue_at_risk
0,Fiber Optic,3035,1236,40.72,3021108.08
1,Cable,830,213,25.66,307794.16
2,DSL,1652,307,18.58,312450.39
3,No Internet Service,1526,113,7.40,43107.19


## 10. Churn by Satisfaction Score

In [14]:
satisfaction_analysis=run_query("""SELECT satisfaction_score,COUNT(*) customers,SUM(churn_label='Yes') churned_customers,ROUND(100*SUM(churn_label='Yes')/COUNT(*),2) churn_rate_pct FROM fact_customer_status GROUP BY satisfaction_score ORDER BY satisfaction_score;""")
satisfaction_analysis

,satisfaction_score,customers,churned_customers,churn_rate_pct
0,1,922,922,100.00
1,2,518,518,100.00
2,3,2665,429,16.10
3,4,1789,0,0.00
4,5,1149,0,0.00


## 11. Revenue-at-Risk by Contract

In [15]:
contract_revenue=run_query("""SELECT a.contract,SUM(f.total_revenue) total_revenue,SUM(CASE WHEN f.churn_label='Yes' THEN f.total_revenue ELSE 0 END) revenue_at_risk FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id GROUP BY a.contract ORDER BY revenue_at_risk DESC;""")
contract_revenue

,contract,total_revenue,revenue_at_risk
0,Month-to-Month,6162488.22,2490105.85
1,One Year,6171794.31,858489.80
2,Two Year,9036849.16,335864.17


## 12. CASE — Customer Value Segmentation

In [16]:
value_segments=run_query("""SELECT customer_id,cltv,CASE WHEN cltv>=5000 THEN 'High Value' WHEN cltv>=3500 THEN 'Medium Value' ELSE 'Low Value' END value_segment FROM fact_customer_status ORDER BY cltv DESC;""")
value_segments.head(10)

,customer_id,cltv,value_segment
0,7622-FWGEW,6500,High Value
1,0383-CLDDA,6499,High Value
2,6024-RUGGH,6499,High Value
3,2683-JXWQQ,6495,High Value
4,4114-QMKVN,6494,High Value
5,8894-JVDCV,6494,High Value
6,1658-XUHBX,6492,High Value
7,2181-TIDSV,6492,High Value
8,4531-AUZNK,6492,High Value
9,0675-NCDYU,6491,High Value


## 13. CTE — High-Value Customer Risk

In [17]:
high_value_summary=run_query("""WITH high_value_customers AS (SELECT customer_id,cltv,total_revenue,satisfaction_score,churn_label FROM fact_customer_status WHERE cltv>=5000) SELECT COUNT(*) high_value_customers,SUM(churn_label='Yes') high_value_churned,ROUND(100*SUM(churn_label='Yes')/COUNT(*),2) high_value_churn_rate_pct FROM high_value_customers;""")
high_value_summary

,high_value_customers,high_value_churned,high_value_churn_rate_pct
0,2574,555,21.56


## 14. Window Function — CLTV Ranking

In [18]:
cltv_rankings=run_query("""SELECT customer_id,cltv,RANK() OVER(ORDER BY cltv DESC) cltv_rank FROM fact_customer_status ORDER BY cltv_rank LIMIT 25;""")
cltv_rankings

,customer_id,cltv,cltv_rank
0,7622-FWGEW,6500,1
1,0383-CLDDA,6499,2
2,6024-RUGGH,6499,2
3,2683-JXWQQ,6495,4
4,4114-QMKVN,6494,5
5,8894-JVDCV,6494,5
6,1658-XUHBX,6492,7
7,2181-TIDSV,6492,7
8,4531-AUZNK,6492,7
9,0675-NCDYU,6491,10


## 15. Window Function — Contract Revenue Share

In [19]:
contract_window=run_query("""SELECT f.customer_id,a.contract,f.total_revenue,SUM(f.total_revenue) OVER(PARTITION BY a.contract) contract_revenue,ROUND(100*f.total_revenue/SUM(f.total_revenue) OVER(PARTITION BY a.contract),2) contract_revenue_share_pct FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id ORDER BY contract_revenue DESC,f.total_revenue DESC LIMIT 50;""")
contract_window.head(20)

,customer_id,contract,total_revenue,contract_revenue,contract_revenue_share_pct
0,0164-APGRB,Two Year,11979.34,9036849.16,0.13
1,8263-QMNTJ,Two Year,11868.34,9036849.16,0.13
2,5451-YHYPW,Two Year,11795.78,9036849.16,0.13
3,3810-DVDQQ,Two Year,11688.90,9036849.16,0.13
4,7569-NMZYQ,Two Year,11634.53,9036849.16,0.13
5,3963-RYFNS,Two Year,11596.99,9036849.16,0.13
6,9351-HXDMR,Two Year,11564.37,9036849.16,0.13
7,5914-XRFQB,Two Year,11529.54,9036849.16,0.13
8,0619-OLYUR,Two Year,11514.81,9036849.16,0.13
9,3508-CFVZL,Two Year,11501.82,9036849.16,0.13


## 16. High-Value Churned Customers

In [20]:
high_value_churn=run_query("""SELECT f.customer_id,f.cltv,f.total_revenue,f.monthly_charge,f.churn_score,f.satisfaction_score,a.contract,a.offer,s.internet_type,s.premium_tech_support FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id JOIN dim_services s ON f.customer_id=s.customer_id WHERE f.churn_label='Yes' ORDER BY f.cltv DESC LIMIT 25;""")
high_value_churn

,customer_id,cltv,total_revenue,monthly_charge,churn_score,satisfaction_score,contract,offer,internet_type,premium_tech_support
0,1043-YCUTE,6484,2066.35,25.15,79,3,Two Year,Offer B,No Internet Service,No
1,1323-OOEPC,6481,7361.72,98.40,87,2,Month-to-Month,No Offer,Fiber Optic,No
2,0112-QWPNC,6452,5997.30,84.35,73,2,One Year,Offer B,Cable,Yes
3,5089-IFSDP,6424,7767.97,109.45,89,1,Two Year,Offer B,Fiber Optic,No
4,0406-BPDVR,6405,7181.56,101.50,68,2,One Year,Offer B,Fiber Optic,No
5,4143-HHPMK,6402,6931.36,85.35,91,3,Month-to-Month,No Offer,Fiber Optic,No
6,1891-FZYSA,6363,7373.82,89.95,82,1,Month-to-Month,No Offer,Fiber Optic,No
7,8634-CILSZ,6350,8630.71,104.70,71,3,One Year,Offer A,Fiber Optic,Yes
8,0617-AQNWT,6347,3147.50,47.85,66,1,Two Year,No Offer,DSL,Yes
9,3313-QKNKB,6304,5492.34,85.55,80,3,One Year,Offer B,Fiber Optic,No


## 17. Customer 360

In [21]:
customer_360=run_query("""SELECT f.customer_id,f.customer_status,f.churn_label,f.churn_score,f.cltv,f.monthly_charge,f.total_revenue,f.satisfaction_score,a.contract,a.payment_method,a.offer,s.internet_type,s.premium_tech_support,d.gender,d.age,d.senior_citizen,l.state,l.city FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id JOIN dim_services s ON f.customer_id=s.customer_id JOIN dim_demographics d ON f.customer_id=d.customer_id JOIN dim_location l ON f.customer_id=l.customer_id;""")
customer_360.head()

,customer_id,customer_status,churn_label,churn_score,cltv,monthly_charge,total_revenue,satisfaction_score,contract,payment_method,offer,internet_type,premium_tech_support,gender,age,senior_citizen,state,city
0,0002-ORFBO,Stayed,No,65,2205,65.60,974.81,3,One Year,Credit Card,No Offer,Cable,Yes,Female,37,No,California,Frazier Park
1,0003-MKNFE,Stayed,No,66,5414,59.90,610.28,5,Month-to-Month,Credit Card,No Offer,Cable,No,Male,46,No,California,Glendale
2,0004-TLHLJ,Churned,Yes,71,4479,73.90,415.45,1,Month-to-Month,Bank Withdrawal,Offer E,Fiber Optic,No,Male,50,No,California,Costa Mesa
3,0011-IGKFF,Churned,Yes,91,3714,98.00,1599.51,1,Month-to-Month,Bank Withdrawal,Offer D,Fiber Optic,No,Male,78,Yes,California,Martinez
4,0013-EXCHZ,Churned,Yes,68,3464,83.90,289.54,1,Month-to-Month,Credit Card,No Offer,Fiber Optic,Yes,Female,75,Yes,California,Camarillo


## 18. Export a Reusable SQL Result

In [22]:
from pathlib import Path
output_path=Path(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 3 — SQL Data Modeling & Analytical Warehouse Design\outputs"); output_path.mkdir(exist_ok=True)
customer_360.to_csv(output_path/"customer_360_export.csv",index=False)
print(f"Saved {len(customer_360):,} Customer 360 rows.")

Saved 7,043 Customer 360 rows.


# Phase 3.3 Conclusion

I have validated the MySQL model and used actual SQL for KPI reconciliation, revenue risk, contract/payment/service analysis, value segmentation, CTEs, window functions, high-value churn analysis, and Customer 360.

**Next:** reusable views, indexes, query-plan inspection, and final documentation.